Data Processing

In [ ]:
import pandas as pd
Patients = pd.read_csv('patients.csv')
Diagnosis = pd.read_csv('diagnoses.csv')
Outcomes = pd.read_csv('outcomes.csv')
Labs = pd.read_csv('labs.csv')

In [ ]:
Patients = Patients.merge(Diagnosis, on='DiagnosisID')
Patients = Patients.merge(Outcomes, on='OutcomeID')

In [ ]:
Patients['AdmissionDate'] = pd.to_datetime(Patients['AdmissionDate'])
Patients['DischargeDate'] = pd.to_datetime(Patients['DischargeDate'])
Patients['LengthOfStay'] = (Patients['DischargeDate'] - Patients['AdmissionDate']).dt.days

In [ ]:
Patients['OutcomeEmcoded'] = Patients['OutcomeName'].map({'Recovered':0, 'Complicated':1, 'Deceased':1})

Creating High Risk Indicator

In [ ]:
import numpy as np

Patients['HighRisk'] = np.where((Patients['Age'] > 65) | (Patients['OutcomeName'].isin(['Complicated', 'Deceased'])),1,0)

In [ ]:
abnormal_conditions = {
    'Blood Sugar': lambda x: x>120,
    'Cholesterol': lambda x: x>200,
    'Hemoglobin': lambda x: x<13
}

def count_abnormal_labs(patient_id):
    patient_labs = Labs[Labs['PatientID'] == patient_id]
    count = 0
    for test_name, condition in abnormal_conditions.items():
        test_results = patient_labs[patient_labs['TestName'] == test_name]
        count += test_results['Result'].apply(condition).sum()
    return count

Patients['AbnormalLabCount'] = Patients['PatientID'].apply(count_abnormal_labs)

Model Training


In [ ]:
feature = Patients[['Age', 'LengthOfStay', 'TreatmentCost']]
target = Patients['OutcomeEmcoded']

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(feature, target, test_size= 0.2, random_state= 42)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression (max_iter= 500)
model.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(x_test)
print("Classification Report: ")
print(classification_report(y_test,y_pred))

Roc Curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
y_prob = model.predict_proba(x_test)[:,1]
fpr ,tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr,tpr)
plt.figure()
plt.plot( fpr, tpr, color= 'blue', label= f'Roc Curve(area = {roc_auc : .2f})')
plt.plot([0,1], [0,1], color= 'gray', linestyle= '--')
plt.xlabel('false positive rate')
plt.ylabel('true positive rate')
plt.title('Receiver Operating Characteristics')
plt.legend(loc = "lower right")
plt.show()

In [ ]:
import joblib

joblib.dump(model, "risk_model.pkl")